# 10 — Deploy PolicyPal to Model Serving

Wraps the
PolicyPal-shape chain (built inline in the c1301 evaluation notebook) into a PyFunc,
registers it to Unity Catalog, creates a Custom Model Serving
endpoint, attaches AI Gateway config (rate limit + inference
table + usage tracking), queries it with Sarah Chen's appeal
question, demonstrates the A/B `traffic_config` split, then directs
you to stop the endpoint from the Serving UI when you're done.

The chain *body* (prompt template + retrieval shape + LLM call)
runs the same retrieval/prompt/LLM logic as the inline `policypal_chain` form
(behaviourally equivalent) — same `num_results=4`, same `_safe_invoke` rate-limit guard. The
only thing this notebook adds on top is the deployment surface.

**⚠ Cost notice.** The **billable** action here is *creating* the
Model Serving endpoint, so the create/mutate cells are gated on a
False-by-default flag (`DEPLOY_ENDPOINT`, `CONFIGURE_GATEWAY`,
`ENABLE_TRAFFIC_SPLIT`) — flip those when you mean
to provision. `QUERY_ENDPOINT` and `VERIFY_AB_SPLIT` default to True:
querying an already-running endpoint is cheap (a few inference calls,
no new billable resource), so leaving them on is harmless. When you're
done, stop or delete the endpoint from the Serving UI (§8) so the
workspace stops billing for idle replicas.
The endpoint can be inspected with below UI location.

![image_1780125784228.png](./image_1780125784228.png "image_1780125784228.png")

**Prerequisites**:

1. The chunking + index-build notebooks have run: `policy_chunks` Delta + the
   `policy_chunks_index` Vector Search index are populated.
2. A Vector Search endpoint hosts the index (`pawshield-vs` by
   default).
3. `databricks-meta-llama-3-1-8b-instruct` pay-per-token FM
   endpoint is available.

**Source verification (May 2026)**:

* `mlflow.models.set_model` + `mlflow.pyfunc.log_model(python_model=<path>)`
  code-based logging — [log-agent docs](https://docs.databricks.com/aws/en/generative-ai/agent-framework/log-agent).
* `w.serving_endpoints.create_and_wait`, `update_config`,
  `put_ai_gateway`, `query`, `delete` — verified against the
  [serving-endpoints SDK reference](https://docs.databricks.com/aws/en/machine-learning/model-serving/create-manage-serving-endpoints).
* `AiGatewayInferenceTableConfig`, `AiGatewayRateLimit` (+
  `AiGatewayRateLimitKey` / `AiGatewayRateLimitRenewalPeriod`
  enums), `AiGatewayUsageTrackingConfig` — SDK class shapes
  verified against the [Databricks SDK serving
  reference](https://databricks-sdk-py.readthedocs.io/en/latest/dbdataclasses/serving.html);
  UI-side configuration flow described in the [AI Gateway config
  docs](https://docs.databricks.com/aws/en/ai-gateway/configure-ai-gateway-endpoints).
  (Guardrails — PII Redaction/Blocking, Unsafe Content, Jailbreak,
  Hallucination, Custom — are intentionally omitted; per the
  overview-serving-endpoints feature-support matrix they are
  supported on External Models + FM-API pay-per-token endpoints
  ONLY, not on Custom Model Serving endpoints like PolicyPal.)
* `DatabricksServingEndpoint` + `DatabricksVectorSearchIndex`
  resource declarations — see [resource auth docs](https://docs.databricks.com/aws/en/generative-ai/agent-framework/log-agent#auth).

In [0]:
# Pin mlflow<3.13 — 3.13.0 has a regression where eval_item.trace is None
# during mlflow.genai.evaluate (trace not associated with eval_request_id).
# Pin databricks-vectorsearch<0.74 — 0.74 renamed VectorSearchIndex → AISearchIndex
# but databricks-ai-bridge still imports the old name, breaking databricks-langchain.
%pip install --quiet --force-reinstall "mlflow[databricks]>=3.12,<3.13" databricks-sdk "databricks-vectorsearch<0.74" databricks-langchain

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jupyter-server 1.23.4 requires anyio<4,>=3.1.0, but you have anyio 4.13.0 which is incompatible.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
dbutils.widgets.text("catalog", "genaicert")
dbutils.widgets.text("vs_endpoint", "pawshield-vs")
dbutils.widgets.text("index_name", "policy_chunks_index")
dbutils.widgets.text("llm_endpoint", "databricks-meta-llama-3-1-8b-instruct")
dbutils.widgets.text("serving_endpoint", "policypal-chain")
catalog = dbutils.widgets.get("catalog")
vs_endpoint = dbutils.widgets.get("vs_endpoint")
index_short = dbutils.widgets.get("index_name")
llm_endpoint = dbutils.widgets.get("llm_endpoint")
serving_endpoint_name = dbutils.widgets.get("serving_endpoint")
index_name = f"{catalog}.pawshield.{index_short}"
registered_model = f"{catalog}.pawshield.policypal_chain"
print(f"Catalog:           {catalog}")
print(f"VS endpoint:       {vs_endpoint}")
print(f"Index:             {index_name}")
print(f"LLM endpoint:      {llm_endpoint}")
print(f"Serving endpoint:  {serving_endpoint_name}")
print(f"Registered model:  {registered_model}")

Catalog:           genaicert
VS endpoint:       pawshield-vs
Index:             genaicert.pawshield.policy_chunks_index
LLM endpoint:      databricks-meta-llama-3-1-8b-instruct
Serving endpoint:  policypal-chain
Registered model:  genaicert.pawshield.policypal_chain


In [0]:
# Serving-endpoint mutations are ASYNC. `put_ai_gateway`, `update_config`,
# `put_traffic_config`, and `delete` all return the moment the API
# accepts the request — the endpoint then transitions through
# `state.config_update = IN_PROGRESS` for 30 s – 5 min while the new
# config propagates to replicas. Without polling, a "now serving v19"
# print fires before the endpoint actually serves v19, which is
# confusing when a follow-up query hits the old replicas.
#
# This helper polls `serving_endpoints.get(name)` until BOTH conditions
# hold: `state.config_update == NOT_UPDATING` AND `state.ready ==
# READY`. It prints one progress line per poll so the cell doesn't look
# frozen. Raises `TimeoutError` if the endpoint hasn't converged within
# `max_wait_s`; callers should re-run after checking the UI in that case.
#
# References (verified May 2026):
#   https://docs.databricks.com/aws/en/machine-learning/model-serving/manage-serving-endpoints
#   SDK: databricks.sdk.service.serving.EndpointStateConfigUpdate /
#        EndpointStateReady enums
import time
from databricks.sdk.service.serving import (
    EndpointStateConfigUpdate,
    EndpointStateReady,
)


def wait_for_endpoint_ready(
    name: str,
    *,
    w,
    max_wait_s: int = 600,
    poll_s: int = 15,
    reason: str = "config update",
) -> None:
    """Poll a serving endpoint until config propagation finishes."""
    start = time.time()
    last_status = None
    while True:
        ep = w.serving_endpoints.get(name=name)
        cfg = ep.state.config_update
        rdy = ep.state.ready
        elapsed = int(time.time() - start)
        status = (cfg, rdy)
        # Only print when the status string changes, so successful
        # waits don't spam the cell output.
        if status != last_status:
            print(
                f"  [{elapsed:>3}s] {name}: config_update={cfg.value}, "
                f"ready={rdy.value} ({reason})"
            )
            last_status = status
        if (
            cfg == EndpointStateConfigUpdate.NOT_UPDATING
            and rdy == EndpointStateReady.READY
        ):
            print(f"  ✓ {name} converged in {elapsed}s.")
            return
        if elapsed > max_wait_s:
            workspace_host = w.config.host.rstrip("/")
            raise TimeoutError(
                f"{name} did not converge within {max_wait_s}s "
                f"(last status: config_update={cfg.value}, "
                f"ready={rdy.value}). Check the UI at "
                f"{workspace_host}/ml/endpoints/{name} — if it still "
                f"shows 'Updating', the API change is in flight; just "
                f"wait and re-run downstream cells. If it shows "
                f"'Failed', inspect the event log there."
            )
        time.sleep(poll_s)

## 1. Verify the prerequisites are in place

In [0]:
from databricks.vector_search.client import VectorSearchClient

vsc = VectorSearchClient(disable_notice=True)
_probe = vsc.get_index(endpoint_name=vs_endpoint, index_name=index_name).describe()
_ready = _probe.get("status", {}).get("ready")
print(f"VS index {index_name}: ready={_ready}")
assert _ready, (
    f"VS index {index_name} is not ready. Re-run the c0801 builder notebook"
    f" (c0801-build-policy-index) and wait for it to come ONLINE."
)

VS index genaicert.pawshield.policy_chunks_index: ready=True


## 2. Import the PolicyPal chain

The chain definition lives in **`policypal_chain.py`** — a
standalone workspace file in this folder. The chain body
(prompt template, `num_results=4` retrieval, `_safe_invoke`
rate-limit guard) runs the same logic as the inline `policypal_chain` form (behaviourally equivalent).
The standalone file is what `log_model(python_model=...)`
references in §3 — Databricks workspace **notebooks** are
objects without a `.py` extension on disk, so `log_model`
cannot point at the notebook source; it has to point at a real
`.py` workspace file in the same folder. The chain file calls
`mlflow.models.set_model(PolicyPalChain())` at module scope,
which MLflow re-executes at serve time to build a fresh
instance per replica.

V1 vs V2 (the §7 traffic split) is *one chain class with
different prompts* — the prompt template is read from
`model_config["prompt_template"]` at `load_context` time, so
two log_model calls with different configs ship two prompt
versions backed by the same code.

In [0]:
import mlflow

# Workspace files in the notebook's folder are reachable via the file
# system at runtime. `import policypal_chain` loads the standalone .py;
# instantiating gives us a chain we can smoke-test in this notebook
# before log_model packages the same file as a deployed artifact.
import sys
sys.path.insert(0, "/Workspace" + dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get().rsplit("/", 1)[0])

from policypal_chain import PolicyPalChain, DEFAULT_PROMPT_TEMPLATE

# V1 prompt is the module default. V2 fixes a real failure mode caught
# by querying v1 on Sarah's appeal question: when the
# retriever returns Section 4.7 (chronic-condition handling) at rank 1
# AND Section 3.2 (general reimbursement formula) at rank 2, the LLM
# weights Section 3.2 in synthesis and tells Sarah the deductible-per-
# claim reset is expected — the opposite of what Section 4.7 actually
# says. V2 adds a specificity instruction so the more-specific clause
# wins. Both prompts are REGISTERED as versions of `policypal_qa` in
# the MLflow Prompt Registry below (§2.5) for versioning and lineage. As
# shipped, the served chain is in-place-template-only: the driver bakes the
# resolved template into `model_config["prompt_template"]` at log time and
# `load_context` reads it directly — it never calls the Prompt Registry at
# serve time, because a Custom (PyFunc) endpoint's auto-auth SP can't read
# it (a serve-time load_prompt → UPDATE_FAILED). Promotion is a re-log +
# redeploy.
POLICYPAL_PROMPT_V1 = DEFAULT_PROMPT_TEMPLATE
POLICYPAL_PROMPT_V2 = DEFAULT_PROMPT_TEMPLATE.replace(
    "Cite the Section number when you quote a clause.",
    (
        "Cite the Section number when you quote a clause.\n\n"
        "When multiple Section excerpts could apply, prioritise the "
        "most specific to the customer's situation: a chronic-condition "
        "clause overrides a general reimbursement clause when the claim "
        "involves a chronic condition; a pet-specific clause overrides a "
        "generic policy clause when the situation is pet-specific."
    ),
)

print(f"PolicyPalChain imported from policypal_chain.py")
print(f"V1 prompt length: {len(POLICYPAL_PROMPT_V1)} chars")
print(f"V2 prompt length: {len(POLICYPAL_PROMPT_V2)} chars")

PolicyPalChain imported from policypal_chain.py
V1 prompt length: 315 chars
V2 prompt length: 623 chars


ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "/databricks/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 541, in send_command
    raise Py4JNetworkError("Answer from Java side is empty")
py4j.protocol.Py4JNetworkError: Answer from Java side is empty

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/databricks/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/databricks/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 564, in send_command
    raise Py4JNetworkError(
py4j.protocol.Py4JNetworkError: Error while sending or receiving


## 2.5. Register the chain's prompts in the MLflow Prompt Registry

Two versions registered as `genaicert.pawshield.policypal_qa`:
v1 (the launch prompt) tagged `@champion`; v2 (the specificity
prompt) tagged `@candidate`. Registration makes each prompt a
first-class versioned artefact for lineage and review.

**How the served chain actually resolves the prompt.** This
chain's `load_context` is *in-place-template-only*: it uses the
literal `model_config["prompt_template"]` baked into the logged
model at `log_model` time and never calls the Prompt Registry at
serve time — a Custom (PyFunc) endpoint's auto-auth SP can't read
the Registry, so a serve-time `load_prompt` makes the endpoint fail
to come READY. So the served prompt is fixed at log time — moving the
`@champion` alias in the Registry does NOT change what the live
endpoint serves. Promotion is therefore a re-log + redeploy (which
§7's A/B split shows by baking a different template into each
version), not an alias move alone. The Registry stays the
versioning/lineage source of truth; the baked template is what runs.

In [0]:
PROMPT_NAME = f"{catalog}.pawshield.policypal_qa"

# Each register_prompt call creates a new version even if the template
# text is identical to an existing version — re-running this cell is
# safe but accumulates version numbers. The alias moves point at the
# latest-registered v1 / v2 so the chain always resolves correctly.
v1_registered = mlflow.genai.register_prompt(
    name=PROMPT_NAME,
    template=POLICYPAL_PROMPT_V1,
    commit_message="v1 — launch prompt; cite Section per clause",
)
v2_registered = mlflow.genai.register_prompt(
    name=PROMPT_NAME,
    template=POLICYPAL_PROMPT_V2,
    commit_message="v2 — adds prioritise-most-specific instruction for the chronic-vs-general clause failure",
)
mlflow.genai.set_prompt_alias(
    name=PROMPT_NAME, alias="champion", version=v1_registered.version,
)
mlflow.genai.set_prompt_alias(
    name=PROMPT_NAME, alias="candidate", version=v2_registered.version,
)
print(
    f"Registered {PROMPT_NAME}:\n"
    f"  v{v1_registered.version} (@champion)  — launch prompt\n"
    f"  v{v2_registered.version} (@candidate) — specificity prompt"
)

# NOTE: we deliberately do NOT load_prompt() here. In MLflow 3 a prompt
# loaded in the session is auto-associated with models logged afterward,
# which makes the served endpoint's auto-auth SP try to read the Registry
# at load time → UPDATE_FAILED. The served chain consumes the baked
# template (model_config["prompt_template"]); the Registry holds the
# canonical record + @champion/@candidate aliases for lineage only.

Registered genaicert.pawshield.policypal_qa:
  v19 (@champion)  — launch prompt
  v20 (@candidate) — specificity prompt


## 3. Register the chain to Unity Catalog

`mlflow.set_registry_uri("databricks-uc")` points MLflow's
registry client at Unity Catalog (mandatory on MLflow 2.x;
default on MLflow 3). `log_model(python_model=<path>, ...)`
code-logs this file; `registered_model_name` lands it in UC in
one step. After this cell, `registered_model@champion` is the
pin the deployment uses.

The `resources=[...]` list declares what the served chain needs
to call at runtime. On deploy, Model Serving's automatic
authentication provisions a system-generated service principal for
the endpoint and grants IT short-lived credentials scoped to
exactly these declared resources. That auto-auth SP is distinct
from the human/SP that CREATES the endpoint — the run-as identity
is fixed at creation and can't be changed later. The deployer must
already hold access to every listed resource for the grant to
succeed; without `resources=[...]`, the served chain can connect
to nothing.

In [0]:
from mlflow.models.resources import (
    DatabricksServingEndpoint,
    DatabricksVectorSearchIndex,
)
from mlflow.models.signature import ModelSignature
from mlflow.tracking import MlflowClient
from mlflow.types.schema import Array, ColSpec, DataType, Schema

mlflow.set_registry_uri("databricks-uc")

# `python_model` must point at a real .py FILE in the workspace, not the
# notebook itself. Workspace notebooks uploaded via `--format SOURCE`
# have no `.py` on disk; `policypal_chain.py` is a workspace file in
# this folder (uploaded via `--format AUTO`), so the relative reference
# resolves correctly when MLflow looks for it.
CHAIN_FILE = "policypal_chain.py"

# Unity Catalog rejects log_model calls that don't carry an explicit
# signature (`Model passed for registration did not contain any signature
# metadata`). MLflow can auto-infer one if the chain's `predict` has typed
# hints OR if we pass `input_example` (which would also fire a live FM call
# during logging — we'd rather not). Cheapest path: declare the schema
# directly. Inputs are the three pandas columns Model Serving passes per
# row; outputs are the per-row dict the chain returns.
policypal_signature = ModelSignature(
    inputs=Schema([
        ColSpec(DataType.string, "question"),
        ColSpec(DataType.string, "state"),
        ColSpec(DataType.string, "tier"),
    ]),
    outputs=Schema([
        ColSpec(DataType.string, "answer"),
        ColSpec(Array(DataType.string), "retrieved_chunk_ids"),
        ColSpec(Array(DataType.string), "retrieved_sections"),
        ColSpec(DataType.string, "version_marker"),
    ]),
)

model_config_v1 = {
    "vs_endpoint":      vs_endpoint,
    "index_name":       index_name,
    "llm_endpoint":     llm_endpoint,
    # Bake the resolved prompt text directly into model_config. The served
    # chain reads this and never calls the Prompt Registry — a Custom
    # (PyFunc) endpoint's auto-auth SP can't read it, so a serve-time
    # load_prompt would make the endpoint fail to come READY (UPDATE_FAILED).
    "prompt_template":  POLICYPAL_PROMPT_V1,
    # version_marker is echoed back on every response so the §7b
    # verification loop can tell champion-routed calls apart from
    # candidate-routed calls without waiting on the inference-table lag.
    "version_marker":   "v1-launch",
}

In [0]:
with mlflow.start_run(run_name="policypal_v1_deploy") as run:
    logged = mlflow.pyfunc.log_model(
        python_model=CHAIN_FILE,
        name="policypal_chain",
        registered_model_name=registered_model,
        model_config=model_config_v1,
        signature=policypal_signature,
        resources=[
            DatabricksServingEndpoint(endpoint_name=llm_endpoint),
            DatabricksVectorSearchIndex(index_name=index_name),
        ],
        # Use pip_requirements (explicit full list) rather than
        # extra_pip_requirements. The auto-inferred deps pull in the
        # entire notebook runtime (databricks-connect, pyarrow==8.0,
        # pip<=22.3.1, debugpy, etc.) which causes unresolvable
        # conflicts in the serving container. A minimal explicit list
        # gives the container a clean, conflict-free environment.
        pip_requirements=[
            "mlflow[databricks]>=2.15",
            "databricks-vectorsearch<0.74",
            "databricks-langchain",
            "langchain-core>=0.3",
            "pandas",
        ],
    )
    print(f"Logged:     {logged.model_uri}")
    print(f"Registered: {registered_model} v{logged.registered_model_version}")

# Pin `@champion` to the just-logged version so the deployment cell
# can resolve the alias to a concrete version.
client = MlflowClient()
client.set_registered_model_alias(
    name=registered_model,
    alias="champion",
    version=logged.registered_model_version,
)
champion_version = logged.registered_model_version
print(f"@champion -> v{champion_version}")

🔗 View Logged Model at: https://<workspace>.cloud.databricks.com/ml/experiments/389838105665656/models/m-74074925ddd74f5c9323952209f09cd2?o=<WORKSPACE_ID>
/local_disk0/.ephemeral_nfs/envs/pythonEnv-d83594ab-8ee4-4dfb-abee-ab23842c607e/lib/python3.10/site-packages/mlflow/pyfunc/utils/data_validation.py:187: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(
/local_disk0/.ephemeral_nfs/envs/pythonEnv-d83594ab-8ee4-4dfb-abee-ab23842c607e/lib/python3.10/site-packages/mlflow/pyfunc/__init__.py:3343: UserWarning: An input example was not provided when logging the model. To ensure the model signature functions correctly, specify the `input_example` parameter. See https://mlflow.org/docs/latest/model/signatures.html#model-input-example for more details about the benefits of using input_e

Uploading artifacts:   0%|          | 0/9 [00:00<?, ?it/s]

🔗 Created version '44' of model 'genaicert.pawshield.policypal_chain': https://<workspace>.cloud.databricks.com/explore/data/models/genaicert/pawshield/policypal_chain/version/44?o=<WORKSPACE_ID>


Logged:     models:/m-74074925ddd74f5c9323952209f09cd2
Registered: genaicert.pawshield.policypal_chain v44
@champion -> v44


## 4. Create the serving endpoint

**⚠ Billable from here until §8 teardown.** The endpoint runs
continuously once created. `scale_to_zero_enabled=True` lets it
drop to zero replicas when idle (cold-start ~few seconds on the
first request after idle); the workspace bills only for active
replica-time. `workload_size="Small"` is the lowest concurrency
tier — appropriate for the notebook demo's volume.

`create_and_wait` blocks ~5–10 minutes for the endpoint to come
READY. Run once, then leave it READY for §5–7 testing.

When you're done, stop or delete it from the 'Serving' entry in the left nav; this gated cell recreates it on a later run if needed.

In [0]:
# ──────────────────────────────────────────────────────────────────────
# Gate flag — visible at the top. Flip to True ONLY when you intend
# to create the endpoint. Once True the workspace starts billing for
# the endpoint until §8 teardown.
# ──────────────────────────────────────────────────────────────────────
DEPLOY_ENDPOINT = False

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import (
    EndpointCoreConfigInput,
    ServedEntityInput,
)

config = EndpointCoreConfigInput(
    name=serving_endpoint_name,  # required field on EndpointCoreConfigInput
    served_entities=[
        ServedEntityInput(
            name="champion",
            entity_name=registered_model,
            entity_version=str(champion_version),
            scale_to_zero_enabled=True,
            workload_size="Small",
        ),
    ],
)

In [0]:
# normally you should just stop the end point in the UI to avoid being charged
# if you need to redeploy a new endpoint, you need to drop the below table first
if DEPLOY_ENDPOINT:
    spark.sql(f"DROP TABLE IF EXISTS {catalog}.monitoring.policypal_payload")

In [0]:
if not DEPLOY_ENDPOINT:
    print(
        f"⏸  Endpoint creation is OFF (DEPLOY_ENDPOINT = False).\n"
        f"   To CREATE a fresh '{serving_endpoint_name}' endpoint (5–10 min,\n"
        f"   starts billing), set  DEPLOY_ENDPOINT = True  at the top of\n"
        f"   the gate-flag cell above and re-run.\n"
        f"   ⚠  Don't use this cell to re-deploy an EXISTING endpoint —\n"
        f"      that errors with ResourceConflict. Use §4a's\n"
        f"      REPOINT_TO_CHAMPION flow instead."
    )
else:
    from databricks.sdk.service.serving import EndpointStateConfigUpdate, EndpointStateReady
    w = WorkspaceClient()
    import time
    # If endpoint exists in a FAILED state, delete and wait until fully gone.
    try:
        ep = w.serving_endpoints.get(name=serving_endpoint_name)
        if ep.state.config_update == EndpointStateConfigUpdate.UPDATE_FAILED:
            print(f"Endpoint {serving_endpoint_name} is in UPDATE_FAILED state — deleting...")
            w.serving_endpoints.delete(name=serving_endpoint_name)
            # Poll until the endpoint is actually gone (max 2 min)
            for i in range(24):
                time.sleep(5)
                try:
                    w.serving_endpoints.get(name=serving_endpoint_name)
                except Exception:
                    print(f"  Endpoint deleted after {(i+1)*5}s.")
                    break
            else:
                raise RuntimeError("Endpoint still exists after 2 min — check the Serving UI.")
    except Exception as e:
        if "RESOURCE_DOES_NOT_EXIST" not in str(e) and "UPDATE_FAILED" not in str(e):
            if "still exists" in str(e) or "RuntimeError" in str(e.__class__.__name__):
                raise
        pass  # endpoint doesn't exist — proceed to create
    print(f"Creating endpoint {serving_endpoint_name} (5–10 min)...")
    w.serving_endpoints.create_and_wait(
        name=serving_endpoint_name,
        config=config,
    )
    print(f"Endpoint {serving_endpoint_name} is READY.")

Endpoint policypal-chain is in UPDATE_FAILED state — deleting...
  Endpoint deleted after 5s.
Creating endpoint policypal-chain (5–10 min)...
Endpoint policypal-chain is READY.


## 5. Attach AI Gateway config (rate limit + inference table + PII)

**⚠ Requires the endpoint to be READY from §4.** Three things
enabled in one update:

* **Rate limit** — 60 calls per minute per user. The exam-relevant
  distinction is that AI Gateway sits in the request path; app-layer
  rate limits miss direct-API consumers.
* **Usage tracking** — populates `system.ai_gateway.usage` rows the
  the monitoring queries read.
* **Inference table** — lands every request + response in
  `genaicert.monitoring.policypal_*` Delta tables for evaluation and monitoring.

**Guardrails note.** AI Gateway guardrails (PII Redaction, Unsafe
Content, Jailbreak, Hallucination, Custom — per the
`ai-gateway/guardrails` doc) are supported on **External model**
endpoints and **Foundation Model APIs pay-per-token** endpoints
ONLY — per the overview-serving-endpoints feature-support matrix,
they are NOT supported on FM-API provisioned throughput,
Databricks Agents, or Custom Model Serving endpoints (PolicyPal
is the last of those). PII + safety enforcement for custom chains
belongs in the chain code (preprocess input / postprocess output
inside the PyFunc); PII/safety handling lives inside the chain code.

**Reset path (when you change the prefix or want a clean run).**
Per the AI Gateway inference-tables doc, *"Specifying an existing
table is not supported. Databricks automatically creates a new
inference table when you create an endpoint or update the Unity
AI Gateway configuration with inference table config enabled,"*
and the table *"may stop logging or become corrupted if users
alter the schema, rename the table, or delete it."* Together these
say: do **not** `DROP TABLE` next to this configure cell while the
gateway is still writing. The safe reset is the gated 3-step cell
at §5b below — disable inference-table config → drop the
quiesced table → re-enable config so the gateway re-creates a
fresh one.

In [0]:
# ──────────────────────────────────────────────────────────────────────
# Gate flag — visible at the top. Flip to True only after the endpoint
# from §4 is READY (or after §4a has re-pointed an existing one).
# !!! NO NEED TO re-config is you are using a restarted an existing configured endpoint.!!!!
# ──────────────────────────────────────────────────────────────────────
CONFIGURE_GATEWAY = False

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import (
    AiGatewayInferenceTableConfig,
    AiGatewayRateLimit,
    AiGatewayRateLimitKey,
    AiGatewayRateLimitRenewalPeriod,
    AiGatewayUsageTrackingConfig,
)

gateway_rate_limits = [
    AiGatewayRateLimit(
        calls=60,
        renewal_period=AiGatewayRateLimitRenewalPeriod.MINUTE,
        key=AiGatewayRateLimitKey.USER,
    ),
]
gateway_usage_tracking = AiGatewayUsageTrackingConfig(enabled=True)
gateway_inference_table = AiGatewayInferenceTableConfig(
    enabled=True,
    catalog_name=catalog,
    schema_name="monitoring",
    table_name_prefix="policypal",
)

In [0]:
if not CONFIGURE_GATEWAY:
    print(
        f"⏸  AI Gateway config is OFF (CONFIGURE_GATEWAY = False).\n"
        f"   To attach rate-limit + usage tracking + inference table to\n"
        f"   '{serving_endpoint_name}', set  CONFIGURE_GATEWAY = True  at\n"
        f"   the top of the gate-flag cell above and re-run.\n"
        f"   ⚠  Guardrails (PII, Unsafe Content, etc.) are intentionally\n"
        f"      omitted here — custom PyFunc endpoints don't support them;\n"
        f"      handle PII/safety inside the chain code instead."
    )
else:
    w = WorkspaceClient()
    # The SDK surface for updating an endpoint's AI Gateway is
    # `serving_endpoints.put_ai_gateway(name, *, rate_limits, ...)` —
    # there is no `update_ai_gateway`.
    #
    # NOTE on guardrails for custom-model endpoints: AI Gateway
    # guardrails (PII Redaction, Unsafe Content, Jailbreak,
    # Hallucination, Custom — per the ai-gateway/guardrails doc)
    # are supported on External model + Foundation Model APIs
    # pay-per-token endpoints ONLY — NOT on FM-API provisioned
    # throughput, Databricks Agents, or Custom Model Serving
    # endpoints (like this PolicyPal PyFunc). Passing a
    # `guardrails=...` kwarg against a custom endpoint yields a
    # NotFound error indicating AI Guardrails isn't supported for
    # that endpoint kind. PII / safety handling for
    # custom-chain endpoints belongs in the chain code itself
    # (preprocess input / postprocess output inside the PyFunc) —
    # PII/safety handling lives inside the chain code.
    w.serving_endpoints.put_ai_gateway(
        name=serving_endpoint_name,
        rate_limits=gateway_rate_limits,
        usage_tracking_config=gateway_usage_tracking,
        inference_table_config=gateway_inference_table,
    )
    wait_for_endpoint_ready(
        serving_endpoint_name, w=w, reason="AI Gateway attach",
    )
    print(f"AI Gateway attached to {serving_endpoint_name} (rate limit + usage + inference table; guardrails not supported on custom endpoints).")

  [  0s] policypal-chain: config_update=IN_PROGRESS, ready=READY (AI Gateway attach)
  [105s] policypal-chain: config_update=NOT_UPDATING, ready=READY (AI Gateway attach)
  ✓ policypal-chain converged in 105s.
AI Gateway attached to policypal-chain (rate limit + usage + inference table; guardrails not supported on custom endpoints).


## 6. Query the endpoint with Sarah's appeal question

The same question used against the inline chain form. The
response shape is identical — `answer` + `retrieved_chunk_ids`
+ `retrieved_sections` — because the chain body is unchanged.

In [0]:
# ──────────────────────────────────────────────────────────────────────
# Gate flag — visible at the top of the cell so the on/off state is
# the first thing you see. Flip to True to actually send a query to
# the deployed endpoint. (The query_policypal helper below is always
# defined regardless of this flag, because §7a reuses it.)
# ──────────────────────────────────────────────────────────────────────
QUERY_ENDPOINT = True

SARAH_APPEAL_QUESTION = (
    "I already paid my deductible earlier this year. Why was "
    "my reimbursement reduced again on Biscuit's ER claim?"
)

In [0]:
from databricks.sdk import WorkspaceClient

# Lazily-built client so the §7a verification loop reuses the same one.
_w_singleton = None


def _w() -> WorkspaceClient:
    global _w_singleton
    if _w_singleton is None:
        _w_singleton = WorkspaceClient()
    return _w_singleton


def query_policypal(question: str, state: str, tier: str) -> dict:
    """Send one PolicyPal question to the deployed endpoint. Returns the
    single per-row prediction dict the chain produced — keys: answer,
    retrieved_chunk_ids, retrieved_sections, version_marker.

    Used by the §6 single-call demo and by the §7a A/B verification
    loop. Encapsulating the call shape here means the loop can iterate
    without copy-pasting the request envelope."""
    response = _w().serving_endpoints.query(
        name=serving_endpoint_name,
        dataframe_records=[{
            "question": question,
            "state": state,
            "tier": tier,
        }],
    )
    return response.predictions[0]

In [0]:
if not QUERY_ENDPOINT:
    print(
        f"⏸  Endpoint query is OFF (QUERY_ENDPOINT = False).\n"
        f"   To send Sarah's appeal question to '{serving_endpoint_name}',\n"
        f"   set  QUERY_ENDPOINT = True  at the top of the gate-flag cell\n"
        f"   above and re-run.\n"
        f"   The query_policypal() helper above is defined regardless;\n"
        f"   §7a's verification loop uses it too."
    )
else:
    pred = query_policypal(
        question=SARAH_APPEAL_QUESTION, state="TX", tier="PawPlus",
    )
    # `.get(..., default)` is defensive: if the endpoint is still
    # serving a model version logged BEFORE the chain file added
    # `version_marker`, the response won't include the key — print
    # a hint instead of crashing with KeyError. To fix: re-run §3
    # (logs a new UC version with the marker) and re-point the
    # endpoint at it via §7 update_config or §4-equivalent
    # serving_endpoints.update_config(name, served_entities=[<new>]).
    marker = pred.get(
        "version_marker",
        "<missing — endpoint is serving an old model version "
        "without version_marker echo; re-log + re-deploy>",
    )
    print("Version marker:    ", marker)
    print("Answer:            ", pred["answer"][:400], "...")
    print("Retrieved chunk ids:", pred["retrieved_chunk_ids"])
    print("Retrieved sections:", pred["retrieved_sections"])

Version marker:     v1-launch
Answer:             According to Section [4.7 Chronic condition exclusions and reimbursement handling], subparagraph (ii), in the event that a claim is initially processed under acute-condition rules but is subsequently determined to relate to a previously documented chronic condition, the Policyholder is entitled to a reimbursement adjustment equal to the deductible amount that was incorrectly deducted.

However, in ...
Retrieved chunk ids: ['pp_plus_tx_v3__chunk_018', 'pp_plus_tx_v3__chunk_006', 'pp_plus_tx_v3__chunk_003', 'pp_plus_tx_v3__chunk_009']
Retrieved sections: ['4.7 Chronic condition exclusions and reimbursement handling', '3.2 Reimbursement calculation', '2.1 Covered services', '4.2 Pre-existing condition exclusion']


## 7. A/B split with `traffic_config`

The documented pattern for comparing two chain versions on the
same endpoint: log v2 with a small prompt tweak, alias as
`@candidate`, then `update_config(served_entities=[v1, v2],
traffic_config={"champion": 90, "candidate": 10})`. One endpoint,
one API contract — clients are unaware of the split.

This cell logs v2 (cheap — just one more model version in UC)
unconditionally, then gates the endpoint update on
`ENABLE_TRAFFIC_SPLIT` since the update reconfigures the
running endpoint.

In [0]:
# Log v2 by re-logging the SAME chain file with a DIFFERENT baked
# model_config["prompt_template"]. One class, two configurations — no
# subclassing, no global-state monkey patching. The cell is
# idempotent; re-running creates a new chain version and re-points
# the served-entities list to it.
model_config_v2 = {
    **model_config_v1,
    # CRITICAL: explicitly override prompt_template — without this, v2
    # inherits V1's baked template from **model_config_v1 and serves the
    # wrong prompt.
    "prompt_template": POLICYPAL_PROMPT_V2,
    "version_marker": "v2-specificity",
}

In [0]:
with mlflow.start_run(run_name="policypal_v2_candidate") as run:
    logged_v2 = mlflow.pyfunc.log_model(
        python_model=CHAIN_FILE,
        name="policypal_chain",
        registered_model_name=registered_model,
        model_config=model_config_v2,
        signature=policypal_signature,
        resources=[
            DatabricksServingEndpoint(endpoint_name=llm_endpoint),
            DatabricksVectorSearchIndex(index_name=index_name),
        ],
        pip_requirements=[
            "mlflow[databricks]>=2.15",
            "databricks-vectorsearch<0.74",
            "databricks-langchain",
            "langchain-core>=0.3",
            "pandas",
        ],
    )

client.set_registered_model_alias(
    name=registered_model,
    alias="candidate",
    version=logged_v2.registered_model_version,
)
candidate_version = logged_v2.registered_model_version
print(f"@candidate -> v{candidate_version}")

🔗 View Logged Model at: https://<workspace>.cloud.databricks.com/ml/experiments/389838105665656/models/m-a18114e8e26d4958a4cbbdf6fb0aece5?o=<WORKSPACE_ID>
/local_disk0/.ephemeral_nfs/envs/pythonEnv-d83594ab-8ee4-4dfb-abee-ab23842c607e/lib/python3.10/site-packages/mlflow/pyfunc/utils/data_validation.py:187: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(
/local_disk0/.ephemeral_nfs/envs/pythonEnv-d83594ab-8ee4-4dfb-abee-ab23842c607e/lib/python3.10/site-packages/mlflow/pyfunc/__init__.py:3343: UserWarning: An input example was not provided when logging the model. To ensure the model signature functions correctly, specify the `input_example` parameter. See https://mlflow.org/docs/latest/model/signatures.html#model-input-example for more details about the benefits of using input_e

Uploading artifacts:   0%|          | 0/9 [00:00<?, ?it/s]

🔗 Created version '45' of model 'genaicert.pawshield.policypal_chain': https://<workspace>.cloud.databricks.com/explore/data/models/genaicert/pawshield/policypal_chain/version/45?o=<WORKSPACE_ID>


@candidate -> v45


In [0]:
# ──────────────────────────────────────────────────────────────────────
# Gate flag — visible at the top. Flip to True to reconfigure the
# running endpoint with the 90/10 split. Requires the endpoint from
# §4 to be READY and both §3 and the v2 log_model above to have run.
# ──────────────────────────────────────────────────────────────────────
ENABLE_TRAFFIC_SPLIT = False
#if you have configured and reuse an existing endpoint, no need to configure anymore

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import (
    ServedEntityInput,
    TrafficConfig,
    Route,
)

ab_served_entities = [
    ServedEntityInput(
        name="champion",
        entity_name=registered_model,
        entity_version=str(champion_version),
        scale_to_zero_enabled=True,
        workload_size="Small",
    ),
    ServedEntityInput(
        name="candidate",
        entity_name=registered_model,
        entity_version=str(candidate_version),
        scale_to_zero_enabled=True,
        workload_size="Small",
    ),
]
ab_traffic_config = TrafficConfig(
    routes=[
        Route(served_entity_name="champion", traffic_percentage=90),
        Route(served_entity_name="candidate", traffic_percentage=10),
    ],
)

In [0]:
if not ENABLE_TRAFFIC_SPLIT:
    print(
        f"⏸  A/B traffic split is OFF (ENABLE_TRAFFIC_SPLIT = False).\n"
        f"   To route 90% of traffic to the @champion version and\n"
        f"   10% to the @candidate version on '{serving_endpoint_name}',\n"
        f"   set  ENABLE_TRAFFIC_SPLIT = True  at the top of the gate-flag\n"
        f"   cell above and re-run.\n"
        f"   Requires §3 (sets champion_version) and §7 top half\n"
        f"   (sets candidate_version) to have run first.\n"
        f"   Then §7a's loop verifies the split immediately via version_marker."
    )
else:
    w = WorkspaceClient()
    # `update_config` takes served_entities + traffic_config as kwargs
    # directly — no EndpointCoreConfigInput wrapper. Route's preferred
    # field is `served_entity_name=` (matches ServedEntityInput.name);
    # `served_model_name=` is the legacy alias.
    print(f"Submitting 90/10 A/B split for {serving_endpoint_name}...")
    w.serving_endpoints.update_config(
        name=serving_endpoint_name,
        served_entities=ab_served_entities,
        traffic_config=ab_traffic_config,
    )
    # The update_config call returns the instant the API accepts the
    # request, but the endpoint is now in IN_PROGRESS while replicas
    # roll the new traffic_config. Querying before this returns can
    # land on the old 100/0 routing — the source of confusing "I just
    # ran it but the split isn't there yet" reports. The wait blocks
    # until both versions are actually being served.
    wait_for_endpoint_ready(
        serving_endpoint_name, w=w,
        reason=f"A/B traffic split → v{champion_version}@90/v{candidate_version}@10",
    )
    print(f"Endpoint {serving_endpoint_name} is now serving v{champion_version} (90%) + v{candidate_version} (10%).")

Submitting 90/10 A/B split for policypal-chain...
  [  0s] policypal-chain: config_update=IN_PROGRESS, ready=READY (A/B traffic split → v44@90/v45@10)
  [527s] policypal-chain: config_update=NOT_UPDATING, ready=READY (A/B traffic split → v44@90/v45@10)
  ✓ policypal-chain converged in 527s.
Endpoint policypal-chain is now serving v44 (90%) + v45 (10%).


## 7a. Verify the A/B split in-process (immediate, no inference-table lag)

The chain echoes a `version_marker` field on every response
(`"v1-launch"` for champion, `"v2-specificity"` for candidate —
the values set in §3 / §7's `model_config`). Looping `query_policypal`
N times and aggregating the markers shows the realised split
immediately, without waiting on the 5–10-minute inference-table
write cycle (§7b is the audit-grade SQL view once that lag has
elapsed).

In [0]:
# ──────────────────────────────────────────────────────────────────────
# Gate flag — visible at the top. Flip to True to send N probe queries
# and aggregate the realised A/B split via the version_marker each
# served entity echoes. No inference-table lag — immediate per-call
# attribution.
# ──────────────────────────────────────────────────────────────────────
VERIFY_AB_SPLIT = True
N_PROBE_REQUESTS = 20

In [0]:
if not VERIFY_AB_SPLIT:
    print(
        f"⏸  A/B verification loop is OFF (VERIFY_AB_SPLIT = False).\n"
        f"   To send {N_PROBE_REQUESTS} probe queries to '{serving_endpoint_name}'\n"
        f"   and print the realised distribution per version_marker,\n"
        f"   set  VERIFY_AB_SPLIT = True  at the top of the gate-flag cell\n"
        f"   above and re-run.\n"
        f"   Requires §7 ENABLE_TRAFFIC_SPLIT to have produced a live split."
    )
else:
    from collections import Counter

    counts: Counter[str] = Counter()
    for i in range(N_PROBE_REQUESTS):
        pred = query_policypal(
            question=SARAH_APPEAL_QUESTION,
            state="TX",
            tier="PawPlus",
        )
        marker = pred.get("version_marker", "unknown")
        counts[marker] += 1
        # Compact per-call line so the loop is observable as it runs.
        print(f"  [{i+1:>2}/{N_PROBE_REQUESTS}] served by: {marker}")

    print()
    print(f"Realised distribution across {N_PROBE_REQUESTS} probes:")
    for marker, n in sorted(counts.items()):
        pct = 100 * n / N_PROBE_REQUESTS
        print(f"  {marker:20s} {n:>3} requests  ({pct:5.1f} %)")
    print()
    print(
        f"Expected from §7 traffic_config: ~90/10 "
        f"(v1-launch / v2-specificity). Each call routes "
        f"independently per the weights, so a 20-probe sample "
        f"will fluctuate around that ratio (sampling error is "
        f"large at N=20). Re-run with larger N for tighter "
        f"convergence."
    )

  [ 1/20] served by: v1-launch
  [ 2/20] served by: v1-launch
  [ 3/20] served by: v1-launch
  [ 4/20] served by: v1-launch
  [ 5/20] served by: v1-launch
  [ 6/20] served by: v1-launch
  [ 7/20] served by: v1-launch
  [ 8/20] served by: v1-launch
  [ 9/20] served by: v1-launch
  [10/20] served by: v1-launch
  [11/20] served by: v1-launch
  [12/20] served by: v1-launch
  [13/20] served by: v1-launch
  [14/20] served by: v1-launch
  [15/20] served by: v1-launch
  [16/20] served by: v2-specificity
  [17/20] served by: v1-launch
  [18/20] served by: v1-launch
  [19/20] served by: v1-launch
  [20/20] served by: v1-launch

Realised distribution across 20 probes:
  v1-launch             19 requests  ( 95.0 %)
  v2-specificity         1 requests  (  5.0 %)

Expected from §7 traffic_config: ~90/10 (v1-launch / v2-specificity). Each call routes independently per the weights, so a 20-probe sample will fluctuate around that ratio (sampling error is large at N=20). Re-run with larger N for tighter

## 7b. Verify the A/B split via the inference table (audit-grade, lag)

The inference table configured in §5 (`monitoring.policypal_payload`,
where `_payload` is the suffix AI Gateway appends to your
`table_name_prefix`) is exactly how you confirm the split is live
and compare answer quality across versions. The key column is
**`served_entity_id`** — the served-entity identifier the
gateway routed each request to (corresponds to the
`ServedEntityInput.name` values `"champion"` / `"candidate"` set
in §4 / §7). This is the column the SQL groups on to confirm
the realised 90/10 split.

Before running the SQL cells below: send a handful of queries
through §6 first (re-run with `QUERY_ENDPOINT = True` 10–20
times, varying the question), then **wait ~5–10 minutes** for
the inference table to catch up — AI Gateway writes it as a
streaming append, not synchronously, so running the SQL
immediately after the first query returns zero rows.

These queries become an A/B-quality-comparison
dashboard that runs the LLM-judge scorers across the
bifurcated traffic, so you can promote @candidate to @champion
on measured-improvement rather than vibes.

In [0]:
# =====================================================================
# HEADS-UP — INFERENCE TABLE LAG (~10 min)
# AI Gateway writes this table as a streaming append, NOT
# synchronously. Rows for a request typically appear ~10 minutes
# after the request was served. Zero rows / low counts running
# these queries IMMEDIATELY after §6 / §7a is EXPECTED — wait
# ~10 minutes and re-run.  For an immediate no-lag view of the
# A/B split, use §7a's in-process loop (reads `version_marker`
# directly from each live response).
# =====================================================================

# Realised traffic split, expressed in the chain's own semantics.
# `response:predictions[0].version_marker` is Databricks SQL JSON-path
# extraction against the `response` JSON column: pull predictions[0]
# (the dict the chain returned for the single input row), then
# the chain-emitted `version_marker` field ("v1-launch" / "v2-specificity"
# from §3 / §7 model_config). Equivalent to GROUPing on
# `served_entity_id` (the platform's own routing label) but the
# chain-emitted marker shows up in the team's product semantics
# rather than as opaque champion/candidate names.

table_name = f"{catalog}.monitoring.policypal_payload"
cols = [c.name for c in spark.table(table_name).schema.fields]
if "response" not in cols:
    print(
        f"⏳ Inference table {table_name} hasn't received data yet.\n"
        f"   The full schema (including 'response') materializes ~10 min\n"
        f"   after the first request is served. Wait and re-run this cell."
    )
else:
    display(spark.sql(f"""
        SELECT
          response:predictions[0].version_marker AS version_marker,
          COUNT(*) AS requests
        FROM {table_name}
        GROUP BY 1
        ORDER BY 1
    """))

version_marker,requests
v1-launch,20
v2-specificity,1


In [0]:
%sql
-- Same ~10-minute inference-table lag applies here too — if this
-- returns zero rows immediately after sending the §6 / §7a queries,
-- wait ~10 min and re-run. The in-process §7a loop is the
-- zero-lag alternative when you just want to see WHICH version
-- handled a call; this SQL is for the side-by-side QUALITY view.

-- Side-by-side answer comparison for Sarah's appeal question.
-- Run after §7 split is live + you've sent ~20 §6 queries with
-- "Biscuit"/"deductible" in the question. `version_marker` (the
-- chain-emitted product label) lets you read v1 vs v2 in the
-- team's own semantics; `served_entity_id` is the platform's
-- routing label — both shown so you can cross-check them.
-- JSON-path note: the endpoint is queried with
--   dataframe_records=[{"question": ..., "state": ..., "tier": ...}]
-- so the stored request shape is
--   {"dataframe_records": [{"question": "...", "state": "...", "tier": "..."}]}
-- The question lives at  request:dataframe_records[0].question  —
-- NOT  request:question  (which resolves to NULL and would
-- silently filter every row out). The response side is shaped
-- {"predictions":[{...}]} so the matching path is
--   response:predictions[0].<field>
SELECT
  request_time,
  response:predictions[0].version_marker AS version_marker,
  served_entity_id,
  request:dataframe_records[0].question::string  AS question,
  response:predictions[0].answer::string         AS answer
FROM IDENTIFIER(:catalog || '.monitoring.policypal_payload')
-- ILIKE needs a STRING operand; the JSON-path expression returns a
-- VARIANT/JSON value, so cast with ::string before the predicate.
-- Without the cast the comparison silently filters every row out.
WHERE request:dataframe_records[0].question::string ILIKE '%Biscuit%'
ORDER BY request_time DESC
LIMIT 20;

request_time,version_marker,served_entity_id,question,answer
2026-06-06T19:10:44.796Z,v1-launch,ff8d32d94bbb433bba1794e2b2f2850d,I already paid my deductible earlier this year. Why was my reimbursement reduced again on Biscuit's ER claim?,"According to Section 3.2 Reimbursement calculation, once the annual deductible is fully paid for a given Policy Year, subsequent claims within that year are not subject to deductible deduction. However, Section 4.7 Chronic condition exclusions and reimbursement handling states that multiple claims arising from a single chronic condition are treated as belonging to ONE condition for deductible purposes. If Biscuit's ER claim is related to a chronic condition, the Policy may be reducing the reimbursement based on that provision. The Policyholder may be entitled to a reimbursement adjustment equal to the deductible amount that was incorrectly deducted, as described in Section 4.7."
2026-06-06T19:10:43.712Z,v1-launch,ff8d32d94bbb433bba1794e2b2f2850d,I already paid my deductible earlier this year. Why was my reimbursement reduced again on Biscuit's ER claim?,"According to Section 3.2 [Reimbursement calculation], the annual deductible is satisfied across all claims filed during the Policy Year on a per-Pet basis. Once the annual deductible is fully paid for a given Policy Year, subsequent claims within that year are not subject to deductible deduction. This suggests that there may not be a credible reason for the reimbursement reduction, as the deductible should have already been satisfied."
2026-06-06T19:10:42.837Z,v1-launch,ff8d32d94bbb433bba1794e2b2f2850d,I already paid my deductible earlier this year. Why was my reimbursement reduced again on Biscuit's ER claim?,"According to Section 3.2, ""the annual deductible is satisfied across all claims filed during the Policy Year on a per-Pet basis"". Since you've already paid your deductible earlier this year, subsequent claims, including Biscuit's ER claim, should not be subject to deductible deduction. However, Section 3.2 also states that the Reimbursable Amount is calculated by subtracting the Outstanding Annual Deductible from the Amount Billed, then applying the Copay Percentage. Without more information, I cannot determine why your reimbursement was reduced."
2026-06-06T19:10:41.849Z,v1-launch,ff8d32d94bbb433bba1794e2b2f2850d,I already paid my deductible earlier this year. Why was my reimbursement reduced again on Biscuit's ER claim?,"According to Section 3.2, Reimbursement calculation, once the annual deductible is fully paid for a given Policy Year, subsequent claims within that year are not subject to deductible deduction. However, this does not guarantee 100% reimbursement. To determine the amount of your reimbursement, we should refer to Section 3.2 again: Reimbursable Amount is calculated as (Amount Billed - Outstanding Annual Deductible) x Copay Percentage. It seems that there might be an adjustment needed for your claim, as it was initially processed under acute-condition rules, but it might relate to a previously documented chronic condition. However, the policy excerpt Section 3 is not clear about this scenario, and for this reason, we cannot accurately determine why your reimbursement was reduced. You may want to request a reimbursement adjustment in accordance with Section 4.7 as the policy does not provide clear insight into this situation."
2026-06-06T19:10:40.502Z,v2-specificity,10483ce7dd30415f8a7734d5989678f8,I already paid my deductible earlier this year. Why was my reimbursement reduced again on Biscuit's ER claim?,"As per Section 3.2 Reimbursement calculation, the annual deductible is satisfied on a per-Pet basis. Since you have already satisfied the annual deductible earlier this year, subsequent claims within that year should not be subject to deductible deduction. However, it's worth noting that Section 4.7 Chronic condition exclusions and reimbursement handling may apply if the claim is related to a chronic condition. Ne

**Gotcha — `ResourceConflict` on rapid config changes.** Reconfiguring
an endpoint that's still in `IN_PROGRESS` from a prior update
(§5 gateway attach, §4 create, or a previous §7 run) returns
`ResourceConflict`. Wait ~1 minute for the prior state transition
to complete (poll `w.serving_endpoints.get(name).state`), then
re-run the cell. The endpoint serialises config writes; there's
no `--force` to override.

## 8. Stop the endpoint

Do this from the Serving UI when you're done, so the workspace stops billing for idle replicas.

![image_1780686303628.png](./image_1780686303628.png "image_1780686303628.png")

## What's next

* The CI/CD notebook c1101 wraps the deploy step — alias-move →
  re-log → endpoint update — so promotion is scripted rather than
  notebook-driven.
* The c1301 evaluation notebook evaluates the *same* chain (inline form) to gate promotion.
  The behaviourally-equivalent chain body means an eval against the inline
  inline chain is a valid proxy for what the served endpoint
  will do.
* The c1401 monitoring notebook reads the inference table created in §5 for production
  monitoring.